In [1]:
# IMPORTS

import os
import gc
import random
import hashlib
import itertools
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import mlflow
import mlflow.pytorch

from helper import AlexNetLike, plot_to_tensorboard, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Usando: cpu


In [2]:
# CARGA Y SPLIT DEL DATASET

data_dir_total = r'data/Split_smol/'
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def get_class(x):
    return x.parent.name

files_totales = []
for x in Path(data_dir_total).rglob('*'):
    if x.is_file() and x.suffix.lower() in valid_extensions:
        try:
            with Image.open(x) as img:
                files_totales.append((x, get_class(x), img.size, img.mode))
        except Exception:
            pass

df_completo = pd.DataFrame(files_totales, columns=["path", "class", "resolution", "mode"])

def calcular_md5(path_objeto):
    hash_md5 = hashlib.md5()
    with open(path_objeto, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

df_completo['md5'] = df_completo['path'].apply(calcular_md5)

fotos_a_eliminar = {"aug_0_F2.large.jpg"}
df_completo = df_completo[
    ~df_completo['path'].apply(lambda p: p.name).isin(fotos_a_eliminar)
].reset_index(drop=True)

df_limpio = df_completo.drop_duplicates(subset=['md5'], keep='first').reset_index(drop=True)

df_train_val, df_test = train_test_split(
    df_limpio, test_size=0.20, stratify=df_limpio['class'], random_state=42
)
df_train, df_val = train_test_split(
    df_train_val, test_size=0.25, stratify=df_train_val['class'], random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

base_train_paths = df_train["path"].apply(str).tolist()
val_image_paths  = df_val["path"].apply(str).tolist()
test_image_paths = df_test["path"].apply(str).tolist()

print(f"Train base: {len(base_train_paths)} | Val: {len(val_image_paths)} | Test: {len(test_image_paths)}")

Train base: 504 | Val: 169 | Test: 169


In [3]:
# DATASET

class CustomImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        self.classes = sorted(list(set([Path(p).parent.name for p in self.image_paths])))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.labels = [self.class_to_idx[Path(p).parent.name] for p in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, label


def build_train_paths_oversampled(base_paths, seed=42):
    """Oversampling para balancear clases en train."""
    random.seed(seed)
    paths = list(base_paths)
    counts = Counter([Path(p).parent.name for p in paths])
    max_count = max(counts.values())
    for cls, count in counts.items():
        faltantes = max_count - count
        if faltantes > 0:
            cls_paths = [p for p in paths if Path(p).parent.name == cls]
            paths.extend(random.choices(cls_paths, k=faltantes))
    return paths


def build_transforms(hp):
    """Construye los transforms a partir de un dict de HPs."""
    size = hp['input_size']
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    aug_list = [A.Resize(size, size)]

    if hp.get('p_hflip', 0) > 0:
        aug_list.append(A.HorizontalFlip(p=hp['p_hflip']))
    if hp.get('p_vflip', 0) > 0:
        aug_list.append(A.VerticalFlip(p=hp['p_vflip']))
    if hp.get('p_rbcontrast', 0) > 0:
        aug_list.append(A.RandomBrightnessContrast(p=hp['p_rbcontrast']))
    if hp.get('p_clahe', 0) > 0:
        aug_list.append(A.CLAHE(p=hp['p_clahe']))
    if hp.get('p_hsv', 0) > 0:
        aug_list.append(A.HueSaturationValue(p=hp['p_hsv']))
    if hp.get('p_rotate', 0) > 0:
        aug_list.append(A.Rotate(limit=20, p=hp['p_rotate']))

    aug_list += [A.Normalize(mean=MEAN, std=STD), ToTensorV2()]

    train_tf = A.Compose(aug_list)
    val_tf   = A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    return train_tf, val_tf


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            out  = model(images)
            loss = criterion(out, labels)
            total_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

In [4]:
# ESPACIO DE BUSQUEDA Y MUESTREO

SEARCH_SPACE = {
    'input_size':   [32, 64],
    'batch_size':   [16, 32],
    'lr':           [1e-3, 1e-4],
    'optimizer':    ['SGD', 'Adam'],
    'momentum':     [0.9, 0.99],      # Solo aplica a SGD
    'weight_decay': [0, 1e-4, 1e-3],
    'dropout':      [0.0, 0.2, 0.3, 0.5],
    'p_hflip':      [0.0, 0.5],
    'p_vflip':      [0.0, 0.5],
    'p_rbcontrast': [0.0, 0.3, 0.5],
    'p_clahe':      [0.0, 0.3],
    'p_hsv':        [0.0, 0.3],
    'p_rotate':     [0.0, 0.4],
}

# Calcular tamaño total del espacio
from functools import reduce
import operator
total = reduce(operator.mul, [len(v) for v in SEARCH_SPACE.values()])
n_samples = max(int(total * 0.05), 50)
print(f"Espacio total: {total:,} combinaciones")
print(f"Se sortean {n_samples} (~5%)")

Espacio total: 36,864 combinaciones
Se sortean 1843 (~5%)


In [5]:
random.seed(99)

def sample_hp():
    hp = {k: random.choice(v) for k, v in SEARCH_SPACE.items()}
    if hp['optimizer'] == 'Adam':
        hp['momentum'] = None  # Adam no usa momentum
    return hp

# Generar n_samples configuraciones únicas
configs = []
seen = set()
while len(configs) < n_samples:
    hp = sample_hp()
    key = str(sorted(hp.items()))
    if key not in seen:
        seen.add(key)
        configs.append(hp)

print(f"Configs generadas: {len(configs)}")

Configs generadas: 1843


In [ ]:
# RANDOM SEARCH

mlflow.set_experiment("CNN_Dermatologia_RandomSearch")

N_EPOCHS_RS  = 30   # Menos épocas para la búsqueda (velocidad vs precisión)
ES_PATIENCE  = 5
NUM_CLASSES  = 9
SEED         = 42

resultados = []

train_paths_os = build_train_paths_oversampled(base_train_paths, seed=SEED)

for i, hp in enumerate(configs):
    run_name = f"rs_{i:03d}"
    print(f"\n[{i+1}/{len(configs)}] {run_name} | {hp}")

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    train_tf, val_tf = build_transforms(hp)

    train_ds = CustomImageDataset(train_paths_os, transform=train_tf)
    val_ds   = CustomImageDataset(val_image_paths, transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=hp['batch_size'], shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=hp['batch_size'], shuffle=False,
                              num_workers=2, pin_memory=True)

    model = AlexNetLike(
        input_size=hp['input_size'],
        dropout=hp['dropout'],
        num_classes=NUM_CLASSES
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if hp['optimizer'] == 'SGD':
        optimizer = optim.SGD(
            model.parameters(), lr=hp['lr'],
            momentum=hp['momentum'], weight_decay=hp['weight_decay']
        )
    else:
        optimizer = optim.Adam(
            model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay']
        )

    best_val_acc = 0
    best_train_acc = 0
    epochs_sin_mejora = 0
    best_path = f"rs_best_{run_name}.pth"

    with mlflow.start_run(run_name=run_name):
        log_hp = {**hp, 'n_params': count_parameters(model), 'seed': SEED}
        mlflow.log_params(log_hp)

        for epoch in range(N_EPOCHS_RS):
            t_loss, t_acc = train_epoch(model, train_loader, optimizer, criterion)
            v_loss, v_acc = evaluate(model, val_loader, criterion)

            mlflow.log_metrics({
                'train_loss': t_loss, 'train_accuracy': t_acc,
                'val_loss':   v_loss, 'val_accuracy':   v_acc
            }, step=epoch)

            if v_acc > best_val_acc:
                best_val_acc   = v_acc
                best_train_acc = t_acc
                epochs_sin_mejora = 0
                torch.save(model.state_dict(), best_path)
            else:
                epochs_sin_mejora += 1
                if epochs_sin_mejora >= ES_PATIENCE:
                    break

        mlflow.log_metrics({
            'best_val_accuracy':   best_val_acc,
            'best_train_accuracy': best_train_acc,
            'gap_train_val':       best_train_acc - best_val_acc
        })

    resultados.append({
        'run': run_name,
        'best_val_acc': best_val_acc,
        'best_train_acc': best_train_acc,
        'gap': best_train_acc - best_val_acc,
        **hp
    })

    # Limpieza
    del model, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    if os.path.exists(best_path):
        os.remove(best_path)

print("\n Random Search finalizado")


[1/1843] rs_000 | {'input_size': 64, 'batch_size': 32, 'lr': 0.001, 'optimizer': 'SGD', 'momentum': 0.9, 'weight_decay': 0, 'dropout': 0.2, 'p_hflip': 0.0, 'p_vflip': 0.5, 'p_rbcontrast': 0.5, 'p_clahe': 0.3, 'p_hsv': 0.0, 'p_rotate': 0.4}


c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# ANALISIS DE RESULTADOS

df_results = pd.DataFrame(resultados)
df_results = df_results.sort_values('best_val_acc', ascending=False).reset_index(drop=True)

print("=== TOP 10 MODELOS ===")
print(df_results[['run', 'best_val_acc', 'best_train_acc', 'gap']].head(10).to_string(index=False))

# Guardar CSV de resultados
df_results.to_csv('resultados_random_search.csv', index=False)
print("\nResultados guardados en resultados_random_search.csv")

NameError: name 'pd' is not defined

In [ ]:
# Filtrar candidatos: val_acc >= 65% y gap <= 10 puntos

candidatos = df_results[
    (df_results['best_val_acc'] >= 65) &
    (df_results['gap'] <= 10)
].reset_index(drop=True)

print(f"Candidatos (val_acc >= 65% y gap <= 10): {len(candidatos)}")
print(candidatos[['run', 'best_val_acc', 'gap', 'optimizer', 'lr',
                  'batch_size', 'input_size', 'dropout', 'momentum']].to_string(index=False))

In [ ]:
# Análisis de qué HPs correlacionan con buen rendimiento

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

hp_categoricos = ['optimizer', 'batch_size', 'input_size', 'dropout',
                  'lr', 'weight_decay', 'momentum', 'p_hflip']

for i, hp_name in enumerate(hp_categoricos):
    if hp_name not in df_results.columns:
        continue
    group = df_results.groupby(hp_name)['best_val_acc'].mean().sort_values(ascending=False)
    axes[i].bar(group.index.astype(str), group.values, color='#f4a7b9', edgecolor='#c0607a')
    axes[i].set_title(f'Val Acc media por {hp_name}', fontsize=10)
    axes[i].set_ylabel('Val Acc (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

plt.suptitle('Efecto promedio de cada HP sobre Val Accuracy', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('hp_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Análisis guardado en hp_analysis.png")

In [ ]:
# Scatterplot val_acc vs gap (buscamos arriba-izquierda: alta acc, bajo gap)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    df_results['gap'], df_results['best_val_acc'],
    c=df_results['best_val_acc'], cmap='RdYlGn', alpha=0.7, s=60
)
plt.colorbar(scatter, ax=ax, label='Val Acc (%)')
ax.axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='gap = 10')
ax.axhline(y=65, color='gray', linestyle=':',  alpha=0.5, label='val_acc = 65%')
ax.set_xlabel('Gap Train-Val (puntos)')
ax.set_ylabel('Val Accuracy (%)')
ax.set_title('Val Accuracy vs Brecha Train-Val')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('scatter_acc_gap.png', dpi=120)
plt.show()